# 🔋 Microgrid Renewable Energy Prediction
## Time Series Analysis & Machine Learning

**Date Range:** January 2015 – December 2018  
**Dataset:** 1,460 daily observations  
**Targets:** Solar generation (MWh) · Wind generation (MWh)  
**Models:** Naive · ARIMA · SARIMA · Holt-Winters · Linear Regression · Random Forest · XGBoost · LSTM · GRU

---
### Table of Contents
1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis (EDA)](#2)
3. [Existing Forecast Comparison](#3)
4. [Time-Series Preprocessing](#4)
5. [Feature Engineering & Train/Val/Test Split](#5)
6. [Model Implementation](#6)
7. [Model Evaluation](#7)
8. [Future Forecasting](#8)


---
## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

# Add src/ to path
SRC_DIR = os.path.join(os.getcwd(), 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
%matplotlib inline

from config import *
from data_loader import load_data, validate_data, get_splits

print('Libraries loaded successfully.')
print(f'  pandas     : {pd.__version__}')
print(f'  numpy      : {np.__version__}')

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = load_data(DATA_PATH)
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
# ── Validate dataset integrity ───────────────────────────────────────────────
summary = validate_data(df)
print('=== Dataset Validation Report ===')
for k, v in summary.items():
    if isinstance(v, dict):
        print(f'  {k}:')
        for kk, vv in v.items():
            if vv > 0: print(f'    {kk}: {vv}')
    else:
        print(f'  {k}: {v}')

---
## 2. Exploratory Data Analysis <a id='2'></a>

In [ ]:
from eda import (
    plot_overview, plot_monthly_boxplots, plot_annual_totals,
    plot_correlation, descriptive_stats, plot_renewable_share, plot_outliers
)

In [ ]:
# 2.1 Descriptive Statistics
print('=== Descriptive Statistics ===')
stats = descriptive_stats(df)
display(stats)

In [ ]:
# 2.2 Time-series overview: solar, wind, demand, renewable share
plot_overview(df)

In [ ]:
# 2.3 Monthly box plots — seasonal patterns
plot_monthly_boxplots(df)

In [ ]:
# 2.4 Annual energy totals
plot_annual_totals(df)

In [ ]:
# 2.5 Correlation heatmap
plot_correlation(df)

In [ ]:
# 2.6 Renewable share time-series & distribution
plot_renewable_share(df)

In [ ]:
# 2.7 Outlier box-plots
plot_outliers(df)

---
## 3. Existing Forecast Comparison <a id='3'></a>

> ⚠️ **Important:** This section evaluates forecasts that were already present in the dataset.
> These are **NOT** generated by our models. They represent a pre-existing operational forecast system.
> Our new models are trained and evaluated separately in Sections 6–7.

In [ ]:
from forecast_comparison import run_all as run_forecast_comparison
existing_metrics = run_forecast_comparison(df)
display(existing_metrics)

---
## 4. Time-Series Preprocessing <a id='4'></a>

In [ ]:
from preprocessing import (
    run_adf_all, plot_rolling_stats, plot_decomposition, plot_acf_pacf
)

# 4.1 ADF Stationarity Test
print('=== ADF Stationarity Tests ===')
adf_results = run_adf_all(df)
display(adf_results[['variable','adf_stat','p_value','critical_5%','stationary']])

In [ ]:
# 4.2 Rolling mean and standard deviation
for col in ['solar_MWh', 'wind_MWh']:
    plot_rolling_stats(df[col], col, window=30)

In [ ]:
# 4.3 Seasonal Decomposition (annual period = 365)
for col in ['solar_MWh', 'wind_MWh']:
    plot_decomposition(df[col], col, period=365, model='additive')

In [ ]:
# 4.4 ACF and PACF — inform ARIMA/SARIMA order selection
for col in ['solar_MWh', 'wind_MWh']:
    plot_acf_pacf(df[col], col, lags=60)

---
## 5. Feature Engineering & Train/Val/Test Split <a id='5'></a>

In [ ]:
# Chronological split — NO shuffling
train, val, test = get_splits(df)

print('Chronological Train / Validation / Test Split')
print(f'  Train : {train.index[0].date()} → {train.index[-1].date()}  ({len(train):,} days)')
print(f'  Val   : {val.index[0].date()} → {val.index[-1].date()}  ({len(val):,} days)')
print(f'  Test  : {test.index[0].date()} → {test.index[-1].date()}  ({len(test):,} days)')
print()
print('Feature engineering strategy:')
print(f'  Lag features    : lag-{LAG_DAYS} days')
print(f'  Rolling windows : {ROLLING_WIN} days (mean + std)')
print(f'  Calendar feats  : day_of_year, month, weekday, weekend, quarter + cyclical encoding')
print(f'  NOT used        : *_forecast_MWh columns, *_kJ columns, renewable_share')

---
## 6. Model Implementation <a id='6'></a>

In [ ]:
from evaluation import evaluate, build_results_table, plot_model_predictions, plot_residuals
import numpy as np

results_all = []  # collect all evaluation rows
all_preds = {'solar_MWh': {}, 'wind_MWh': {}}

### 6.1 Baseline — Naive Forecasting

**Rationale:** Predicts tomorrow = today. Establishes the floor. Any useful model must beat this.

In [ ]:
from models_statistical import naive_forecast

for target in TARGETS:
    preds = naive_forecast(train, val, test, target)
    actual = test[target].values[:len(preds)]
    m = evaluate(actual, preds, 'Naive', target)
    results_all.append(m)
    all_preds[target]['Naive'] = preds
    print(f'Naive [{target}] → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
          f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')

### 6.2 ARIMA

**Rationale:** Classic statistical model for univariate time series. Order (2,1,2) chosen based on ACF/PACF — d=1 for stationarity, p=2, q=2.

In [ ]:
from models_statistical import fit_arima

ARIMA_ORDERS = {'solar_MWh': (2,1,2), 'wind_MWh': (2,1,2)}

for target in TARGETS:
    print(f'Fitting ARIMA for {target}...')
    preds = fit_arima(train, val, test, target, order=ARIMA_ORDERS[target])
    actual = test[target].values[:len(preds)]
    m = evaluate(actual, preds, 'ARIMA', target)
    results_all.append(m)
    all_preds[target]['ARIMA'] = preds
    print(f'  → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
          f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')

### 6.3 SARIMA

**Rationale:** Extends ARIMA with seasonal components. Weekly seasonality (s=7) captures the day-of-week patterns visible in energy consumption.

In [ ]:
from models_statistical import fit_sarima

for target in TARGETS:
    print(f'Fitting SARIMA for {target}...')
    preds = fit_sarima(train, val, test, target,
                       order=(1,1,1), seasonal_order=(1,0,1,7))
    actual = test[target].values[:len(preds)]
    m = evaluate(actual, preds, 'SARIMA', target)
    results_all.append(m)
    all_preds[target]['SARIMA'] = preds
    print(f'  → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
          f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')

### 6.4 Holt-Winters Exponential Smoothing

**Rationale:** Explicitly models level + trend + seasonal components. Annual period (365) aligns with the strong solar and wind seasonal cycles.

In [ ]:
from models_statistical import fit_holtwinters

for target in TARGETS:
    print(f'Fitting Holt-Winters for {target}...')
    preds = fit_holtwinters(train, val, test, target, seasonal_periods=365)
    actual = test[target].values[:len(preds)]
    m = evaluate(actual, preds, 'Holt-Winters', target)
    results_all.append(m)
    all_preds[target]['Holt-Winters'] = preds
    print(f'  → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
          f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')

### 6.5 Linear Regression

**Rationale:** Interpretable ML baseline. Uses lag + calendar + rolling features. Establishes whether a simple linear relationship captures the patterns.

In [ ]:
from models_ml import fit_linear_regression

lr_models = {}
for target in TARGETS:
    preds, idx_te, model, scaler = fit_linear_regression(train, val, test, target)
    actual = test.loc[idx_te, target].values
    m = evaluate(actual, preds, 'Linear Regression', target)
    results_all.append(m)
    all_preds[target]['Linear Regression'] = preds
    lr_models[target] = (model, scaler, idx_te)
    print(f'Linear Regression [{target}] → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
          f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')

### 6.6 Random Forest

**Rationale:** Ensemble method robust to noise and non-linearity. Feature importance reveals which lags and calendar features matter most.

In [ ]:
from models_ml import fit_random_forest, plot_feature_importance

rf_models = {}
for target in TARGETS:
    preds, idx_te, model, importances = fit_random_forest(train, val, test, target)
    actual = test.loc[idx_te, target].values
    m = evaluate(actual, preds, 'Random Forest', target)
    results_all.append(m)
    all_preds[target]['Random Forest'] = preds
    rf_models[target] = (model, idx_te)
    print(f'Random Forest [{target}] → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
          f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')
    plot_feature_importance(importances, 'Random Forest', target)

### 6.7 XGBoost

**Rationale:** Gradient boosted trees — typically the highest-performing tree model. Early stopping on validation prevents overfitting.

In [ ]:
from models_ml import fit_xgboost

xgb_models = {}
for target in TARGETS:
    preds, idx_te, model, importances = fit_xgboost(train, val, test, target)
    actual = test.loc[idx_te, target].values
    m = evaluate(actual, preds, 'XGBoost', target)
    results_all.append(m)
    all_preds[target]['XGBoost'] = preds
    xgb_models[target] = (model, idx_te)
    print(f'XGBoost [{target}] → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
          f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')
    plot_feature_importance(importances, 'XGBoost', target)

### 6.8 LSTM

**Rationale:** Long Short-Term Memory networks capture long-range dependencies in sequential data. Look-back window = 30 days.

In [ ]:
from models_dl import fit_lstm, TF_AVAILABLE

lstm_models = {}
if TF_AVAILABLE:
    for target in TARGETS:
        print(f'Training LSTM for {target}...')
        preds, idx_te = fit_lstm(train, val, test, target)
        if preds is not None:
            actual = test.loc[idx_te, target].values if idx_te is not None else test[target].values[:len(preds)]
            m = evaluate(actual[:len(preds)], preds, 'LSTM', target)
            results_all.append(m)
            all_preds[target]['LSTM'] = preds
            lstm_models[target] = (preds, idx_te)
            print(f'  LSTM [{target}] → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
                  f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')
else:
    print('TensorFlow not available — LSTM skipped.')

### 6.9 GRU

**Rationale:** Gated Recurrent Unit — a lighter alternative to LSTM with fewer parameters, often achieving similar accuracy with faster training.

In [ ]:
from models_dl import fit_gru

gru_models = {}
if TF_AVAILABLE:
    for target in TARGETS:
        print(f'Training GRU for {target}...')
        preds, idx_te = fit_gru(train, val, test, target)
        if preds is not None:
            actual = test.loc[idx_te, target].values if idx_te is not None else test[target].values[:len(preds)]
            m = evaluate(actual[:len(preds)], preds, 'GRU', target)
            results_all.append(m)
            all_preds[target]['GRU'] = preds
            gru_models[target] = (preds, idx_te)
            print(f'  GRU [{target}] → MAE={m["MAE (MWh)"]:,.0f}  RMSE={m["RMSE (MWh)"]:,.0f}  '
                  f'MAPE={m["MAPE (%)"]}%  R²={m["R²"]}')
else:
    print('TensorFlow not available — GRU skipped.')

---
## 7. Model Evaluation <a id='7'></a>

In [ ]:
from evaluation import (
    build_results_table, plot_model_predictions,
    plot_model_comparison, plot_r2_comparison, plot_residuals
)
import os

# Build comparison table
results_df = build_results_table(results_all)

print('═'*70)
print('  MODEL EVALUATION — SOLAR GENERATION')
print('═'*70)
solar_df = results_df[results_df['Target']=='solar_MWh'].reset_index(drop=True)
display(solar_df)

print()
print('═'*70)
print('  MODEL EVALUATION — WIND GENERATION')
print('═'*70)
wind_df = results_df[results_df['Target']=='wind_MWh'].reset_index(drop=True)
display(wind_df)

# Save to CSV
results_df.to_csv(os.path.join(RESULTS_DIR, 'model_results.csv'), index=False)
print(f'\nResults saved to outputs/results/model_results.csv')

In [ ]:
# Best model per target
print('\n=== BEST MODELS (by RMSE) ===')
for target in TARGETS:
    best = results_df[results_df['Target']==target].sort_values('RMSE (MWh)').iloc[0]
    print(f'  {target}: {best["Model"]} — RMSE={best["RMSE (MWh)"]:,.0f} MWh, '
          f'MAPE={best["MAPE (%)"]:.2f}%, R²={best["R²"]:.4f}')

In [ ]:
# Actual vs Predicted plots for each target
for target in TARGETS:
    actual_series = test[target]
    plot_model_predictions(actual_series, all_preds[target], target)

In [ ]:
# Comparison bar charts
for target in TARGETS:
    plot_model_comparison(results_df, target)

In [ ]:
# R² comparison
plot_r2_comparison(results_df)

In [ ]:
# Residual plots
for target in TARGETS:
    actual_arr = test[target].values
    plot_residuals(actual_arr, all_preds[target], target)

---
## 8. Future Forecasting <a id='8'></a>

> 📌 Using the best-performing ML model (Random Forest or XGBoost) for 90-day ahead iterative forecasting.
> Predictions are clearly distinguished from historical observations.

In [ ]:
from future_forecast import iterative_forecast, plot_future_forecast
from models_ml import fit_random_forest, fit_xgboost

HORIZON = 90  # days ahead

# Select best ML model per target (by RMSE on test set)
best_model_name = {}
for target in TARGETS:
    ml_models = ['Linear Regression','Random Forest','XGBoost']
    sub = results_df[(results_df['Target']==target) &
                     (results_df['Model'].isin(ml_models))]
    if not sub.empty:
        best_model_name[target] = sub.sort_values('RMSE (MWh)').iloc[0]['Model']
    else:
        best_model_name[target] = 'Random Forest'

print('Best ML model selected for future forecasting:')
for t, m in best_model_name.items():
    print(f'  {t}: {m}')

In [ ]:
# Re-train best model on ALL available data (train+val+test) for future prediction
full_data = pd.concat([train, val, test])
future_preds = {}

for target in TARGETS:
    model_name = best_model_name[target]
    print(f'Re-training {model_name} on full dataset for {target}...')

    if model_name == 'XGBoost':
        _, _, model, _ = fit_xgboost(train, val, test, target)
        scaler = None
    elif model_name == 'Random Forest':
        _, _, model, _ = fit_random_forest(train, val, test, target)
        scaler = None
    else:  # Linear Regression
        from models_ml import fit_linear_regression
        _, _, model, scaler = fit_linear_regression(train, val, test, target)

    forecast_series = iterative_forecast(model, full_data, target, scaler, horizon=HORIZON)
    future_preds[target] = forecast_series
    print(f'  {target}: {HORIZON}-day forecast generated.')
    print(f'  Range: {forecast_series.values.min():.0f} – {forecast_series.values.max():.0f} MWh')

In [ ]:
# Plot future forecast
demand_mean = df['demand_MWh'].tail(90).mean()  # Use recent demand average for share estimate

forecast_table = plot_future_forecast(
    df,
    solar_future=future_preds['solar_MWh'],
    wind_future=future_preds['wind_MWh'],
    horizon=HORIZON,
    demand_mean=demand_mean,
)
print(f'\n=== Future Forecast Summary ({HORIZON} days) ===')
print(forecast_table.describe().round(1))

# Save forecast
forecast_table.to_csv(os.path.join(RESULTS_DIR, 'future_forecast.csv'), index=False)
print(f'\nFuture forecast saved to outputs/results/future_forecast.csv')
display(forecast_table.head(10))

---
## Summary

| Section | Status |
|---|---|
| Dataset loaded & validated | ✅ |
| EDA (overview, seasonal, correlation, outliers) | ✅ |
| Existing forecast comparison (solar/wind/demand) | ✅ |
| Preprocessing (ADF, decomposition, ACF/PACF) | ✅ |
| Naive baseline | ✅ |
| ARIMA | ✅ |
| SARIMA | ✅ |
| Holt-Winters | ✅ |
| Linear Regression | ✅ |
| Random Forest | ✅ |
| XGBoost | ✅ |
| LSTM | ✅ |
| GRU | ✅ |
| Model comparison & best model selection | ✅ |
| 90-day future forecast | ✅ |

> All outputs (figures, CSVs) are saved in the `outputs/` directory.


---
## 9. Interactive Custom Date-Range Forecaster <a id="9"></a>

> 💡 **Input ANY start date and end date** to forecast Solar, Wind, and Total Renewable generation based on what the models learned from 2015–2018!


In [ ]:
from predict import MicrogridPredictor, plot_prediction_results, display_report

# 👉 ENTER YOUR CUSTOM DATE RANGE HERE:
START_DATE = "2019-01-01"
END_DATE   = "2019-12-31"

predictor = MicrogridPredictor(model_type="ensemble")
predictor.train()
df_custom = predictor.predict_range(START_DATE, END_DATE)

display(df_custom.head(10))

# Plot the custom forecast
import matplotlib.pyplot as plt
from PIL import Image
img_path = f"outputs/predictions/forecast_{START_DATE}_to_{END_DATE}.png"
plot_prediction_results(df_custom, START_DATE, END_DATE, img_path)

# Display the plot
img = Image.open(img_path)
plt.figure(figsize=(14, 11), dpi=100)
plt.imshow(img)
plt.axis("off")
plt.show()
